# 08. Treinamento de Modelos - TimeSeriesSplit & Threshold Financeiro

## 🎯 Objetivo

Implementar **treinamento de modelos de ML** seguindo as **melhores práticas para detecção de fraude**:

1. **TimeSeriesSplit**: Validação cruzada temporal (em vez de CV aleatório)
2. **Balanceamento correto**: SMOTE/RUS **dentro dos folds** de treino
3. **Threshold financeiro**: Otimização baseada em **Matriz de Custo**
4. **Curva Precision-Recall**: Métrica mais adequada que ROC para classes desbalanceadas

## 📚 Melhores Práticas Implementadas

### ✅ Ponto 8 - Balanceamento de Classes
- **SMOTE aplicado dentro dos folds de CV** (nunca no dataset completo)
- Alternativa: `scale_pos_weight` (XGBoost/LightGBM)

### ✅ Ponto 9 - Validação Temporal
- **TimeSeriesSplit** em vez de StratifiedKFold
- Simula cenário real de produção (treino no passado, teste no futuro)

### ✅ Ponto 10 - Threshold Financeiro (CRÍTICO)
- **Matriz de Custo**:
  - FN (False Negative): Custo da fraude não detectada = Valor da transação
  - FP (False Positive): Custo de bloquear cliente legítimo = Custo fixo de fricção
- **Threshold ótimo**: Minimiza custo total (não usa 0.5 padrão)
- **Precision-Recall Curve**: Mais informativa que ROC

---

## 1. Setup - Importações e Configuração

In [ ]:
# Configurar path para importar módulos do projeto
import sys
from pathlib import Path

notebook_dir = Path.cwd()
if notebook_dir.name == 'notebooks':
    sys.path.insert(0, str(notebook_dir.parent))

# Importar configurações do projeto
from source.config import PROJ_ROOT, get_data_path, get_model_path, get_figure_path, ensure_directories

# Garantir que os diretórios existam
ensure_directories()

print(f"✅ Projeto Root: {PROJ_ROOT}")
print(f"✅ Configurações carregadas de source/config.py")

In [ ]:
# Importações padrão
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime
import json
import joblib
from typing import Dict, Tuple

# Scikit-Learn - Validação e Métricas
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve,
    make_scorer
)

# Scikit-Learn - Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# XGBoost e LightGBM
import xgboost as xgb
import lightgbm as lgb

# Imbalanced-Learn
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

# Configurações
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
pd.set_option('display.max_columns', None)

# Seed para reprodutibilidade
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✅ Bibliotecas importadas com sucesso!")

## 2. Carregamento dos Dados Transformados

Carregamos os dados **já transformados** pelo Notebook 06.

In [ ]:
print("="*80)
print("CARREGAMENTO DOS DADOS TRANSFORMADOS")
print("="*80)

# Carregar datasets
X_train = pd.read_csv(get_data_path('X_train.csv', 'processed'))
X_oot = pd.read_csv(get_data_path('X_oot.csv', 'processed'))
y_train = pd.read_csv(get_data_path('y_train.csv', 'processed')).squeeze()
y_oot = pd.read_csv(get_data_path('y_oot.csv', 'processed')).squeeze()

print(f"\n📊 Treino:")
print(f"   X_train: {X_train.shape}")
print(f"   y_train: {y_train.shape}")
print(f"   Taxa de lavagem: {y_train.mean()*100:.2f}%")

print(f"\n📊 OOT:")
print(f"   X_oot: {X_oot.shape}")
print(f"   y_oot: {y_oot.shape}")
print(f"   Taxa de lavagem: {y_oot.mean()*100:.2f}%")

print(f"\n✅ Dados carregados com sucesso!")

## 3. Configuração de Modelos

Definimos os modelos a serem treinados com hiperparâmetros ajustados para classes desbalanceadas.

In [ ]:
# Calcular scale_pos_weight para XGBoost e LightGBM
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()
scale_pos_weight_value = negative_count / positive_count

print(f"📊 Distribuição de Classes:")
print(f"   Negativos (Normal): {negative_count:,}")
print(f"   Positivos (Lavagem): {positive_count:,}")
print(f"   scale_pos_weight: {scale_pos_weight_value:.2f}")

# Dicionário de modelos
models = {
    'Logistic Regression': LogisticRegression(
        random_state=RANDOM_STATE,
        max_iter=1000,
        class_weight='balanced',
        solver='saga',
        n_jobs=-1
    ),
    
    'Random Forest': RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_estimators=100,
        max_depth=10,
        min_samples_split=50,
        min_samples_leaf=20,
        class_weight='balanced',
        n_jobs=-1
    ),
    
    'Gradient Boosting': GradientBoostingClassifier(
        random_state=RANDOM_STATE,
        n_estimators=100,
        max_depth=5,
        learning_rate=0.1,
        subsample=0.8
    ),
    
    'XGBoost': xgb.XGBClassifier(
        random_state=RANDOM_STATE,
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight_value,
        eval_metric='aucpr',
        use_label_encoder=False,
        n_jobs=-1
    ),
    
    'LightGBM': lgb.LGBMClassifier(
        random_state=RANDOM_STATE,
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight_value,
        n_jobs=-1,
        verbose=-1
    )
}

print(f"\n✅ {len(models)} modelos configurados")

## 4. Validação Temporal com TimeSeriesSplit

⚠️ **CRÍTICO**: Usamos TimeSeriesSplit em vez de KFold para respeitar a ordem temporal.

In [ ]:
# Configurar TimeSeriesSplit
n_splits = 5
tscv = TimeSeriesSplit(n_splits=n_splits)

print("="*80)
print("CONFIGURAÇÃO DE VALIDAÇÃO TEMPORAL")
print("="*80)

print(f"\n📅 TimeSeriesSplit com {n_splits} splits")
print(f"\n✓ Cada fold:")

# Visualizar splits
for fold_idx, (train_idx, val_idx) in enumerate(tscv.split(X_train), 1):
    print(f"\n  Fold {fold_idx}:")
    print(f"    Treino: {len(train_idx):,} samples")
    print(f"    Validação: {len(val_idx):,} samples")
    print(f"    Proporção: {len(train_idx)/len(X_train)*100:.1f}% treino / {len(val_idx)/len(X_train)*100:.1f}% val")

## 5. Treinamento com Balanceamento Correto

Aplicamos SMOTE **dentro de cada fold de treino** para evitar data leakage.

In [ ]:
# Função para treinar com SMOTE dentro dos folds
def train_with_temporal_cv(model, X, y, tscv, use_smote=True):
    """
    Treina modelo com TimeSeriesSplit e SMOTE aplicado dentro dos folds.
    
    Args:
        model: Modelo sklearn
        X: Features
        y: Target
        tscv: TimeSeriesSplit configurado
        use_smote: Se True, aplica SMOTE no treino de cada fold
    
    Returns:
        Dict com métricas de validação cruzada
    """
    val_scores = {
        'precision': [],
        'recall': [],
        'f1': [],
        'roc_auc': [],
        'avg_precision': []
    }
    
    for fold_idx, (train_idx, val_idx) in enumerate(tscv.split(X), 1):
        # Dados do fold
        X_fold_train, X_fold_val = X.iloc[train_idx], X.iloc[val_idx]
        y_fold_train, y_fold_val = y.iloc[train_idx], y.iloc[val_idx]
        
        # Aplicar SMOTE APENAS no treino do fold
        if use_smote:
            smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
            X_fold_train, y_fold_train = smote.fit_resample(X_fold_train, y_fold_train)
        
        # Treinar modelo
        model.fit(X_fold_train, y_fold_train)
        
        # Predições
        y_pred_proba = model.predict_proba(X_fold_val)[:, 1]
        y_pred = (y_pred_proba >= 0.5).astype(int)
        
        # Calcular métricas
        val_scores['precision'].append(precision_score(y_fold_val, y_pred, zero_division=0))
        val_scores['recall'].append(recall_score(y_fold_val, y_pred, zero_division=0))
        val_scores['f1'].append(f1_score(y_fold_val, y_pred, zero_division=0))
        val_scores['roc_auc'].append(roc_auc_score(y_fold_val, y_pred_proba))
        val_scores['avg_precision'].append(average_precision_score(y_fold_val, y_pred_proba))
    
    # Médias
    avg_scores = {metric: np.mean(scores) for metric, scores in val_scores.items()}
    
    return avg_scores

print("✅ Função de treinamento com validação temporal definida")

In [ ]:
# Treinar todos os modelos com TimeSeriesSplit
print("="*80)
print("TREINAMENTO COM TIMESERIESPLIT + SMOTE")
print("="*80)

cv_results = {}

for model_name, model in models.items():
    print(f"\n🔄 Treinando {model_name}...")
    
    # Validação cruzada temporal
    scores = train_with_temporal_cv(model, X_train, y_train, tscv, use_smote=True)
    
    cv_results[model_name] = scores
    
    print(f"\n  ✅ {model_name} - Validação Cruzada (média {n_splits} folds):")
    print(f"     Precision: {scores['precision']:.4f}")
    print(f"     Recall: {scores['recall']:.4f}")
    print(f"     F1-Score: {scores['f1']:.4f}")
    print(f"     ROC-AUC: {scores['roc_auc']:.4f}")
    print(f"     Avg Precision: {scores['avg_precision']:.4f}")

print("\n" + "="*80)
print("✅ TREINAMENTO CONCLUÍDO")
print("="*80)

## 6. Seleção do Melhor Modelo

Escolhemos o modelo com melhor Average Precision (métrica mais adequada para classes desbalanceadas).

In [ ]:
# Criar DataFrame com resultados
df_cv_results = pd.DataFrame(cv_results).T
df_cv_results = df_cv_results.sort_values('avg_precision', ascending=False)

print("="*80)
print("RANKING DE MODELOS (por Average Precision)")
print("="*80)
print(df_cv_results)

# Melhor modelo
best_model_name = df_cv_results.index[0]
print(f"\n🏆 Melhor Modelo: {best_model_name}")
print(f"   Average Precision: {df_cv_results.loc[best_model_name, 'avg_precision']:.4f}")

In [ ]:
# Visualizar comparação
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Average Precision
df_cv_results['avg_precision'].sort_values().plot(
    kind='barh', ax=axes[0], color='steelblue'
)
axes[0].set_title('Average Precision (Validação Cruzada)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Average Precision')
axes[0].grid(alpha=0.3)

# Plot 2: F1-Score
df_cv_results['f1'].sort_values().plot(
    kind='barh', ax=axes[1], color='coral'
)
axes[1].set_title('F1-Score (Validação Cruzada)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('F1-Score')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(get_figure_path('model_comparison_cv.png'))
plt.show()

print(f"\n📊 Gráfico salvo em: {get_figure_path('model_comparison_cv.png')}")

## 7. Treinamento Final do Melhor Modelo

Treinamos o melhor modelo no **X_train completo** (com SMOTE) para avaliar no OOT.

In [ ]:
print("="*80)
print("TREINAMENTO FINAL - MELHOR MODELO")
print("="*80)

# Aplicar SMOTE no treino completo
print("\n🔄 Aplicando SMOTE no dataset de treino completo...")
smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"   Original: {X_train.shape[0]:,} samples")
print(f"   Balanceado: {X_train_balanced.shape[0]:,} samples")
print(f"   Taxa de lavagem após SMOTE: {y_train_balanced.mean()*100:.2f}%")

# Treinar melhor modelo
best_model = models[best_model_name]
print(f"\n🔄 Treinando {best_model_name} no dataset balanceado...")
best_model.fit(X_train_balanced, y_train_balanced)

print(f"\n✅ {best_model_name} treinado com sucesso!")

## 8. Threshold Financeiro - Matriz de Custo (CRÍTICO)

⚠️ **NÃO usamos threshold 0.5 padrão!**  
Otimizamos o threshold baseado na **Matriz de Custo** do negócio.

In [ ]:
# Definir custos (ajustar conforme negócio)
COST_FN = 1000  # Custo de não detectar uma fraude (ex: valor médio de transação fraudulenta)
COST_FP = 50    # Custo de bloquear um cliente legítimo (custo de fricção/insatisfação)

print("="*80)
print("OTIMIZAÇÃO DE THRESHOLD - MATRIZ DE CUSTO")
print("="*80)

print(f"\n💰 Custos Definidos:")
print(f"   False Negative (Fraude não detectada): ${COST_FN}")
print(f"   False Positive (Cliente bloqueado): ${COST_FP}")

# Predições de probabilidade no OOT
y_oot_proba = best_model.predict_proba(X_oot)[:, 1]

# Testar diferentes thresholds
thresholds = np.arange(0.05, 0.96, 0.01)
costs = []
precisions = []
recalls = []

for threshold in thresholds:
    y_oot_pred = (y_oot_proba >= threshold).astype(int)
    
    # Confusion matrix
    tn, fp, fn, tp = confusion_matrix(y_oot, y_oot_pred).ravel()
    
    # Custo total
    total_cost = (fn * COST_FN) + (fp * COST_FP)
    costs.append(total_cost)
    
    # Métricas
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    precisions.append(precision)
    recalls.append(recall)

# Encontrar threshold ótimo (custo mínimo)
optimal_idx = np.argmin(costs)
optimal_threshold = thresholds[optimal_idx]
optimal_cost = costs[optimal_idx]

print(f"\n🎯 Threshold Ótimo: {optimal_threshold:.3f}")
print(f"   Custo Total: ${optimal_cost:,.2f}")
print(f"   Precision: {precisions[optimal_idx]:.4f}")
print(f"   Recall: {recalls[optimal_idx]:.4f}")

# Comparar com threshold padrão 0.5
default_idx = np.argmin(np.abs(thresholds - 0.5))
default_cost = costs[default_idx]

print(f"\n📊 Comparação com Threshold Padrão (0.5):")
print(f"   Custo com threshold 0.5: ${default_cost:,.2f}")
print(f"   Economia com threshold otimizado: ${default_cost - optimal_cost:,.2f}")
print(f"   Redução de custo: {(default_cost - optimal_cost)/default_cost*100:.2f}%")

In [ ]:
# Visualizar curva de custo
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Custo Total vs Threshold
axes[0].plot(thresholds, costs, linewidth=2, color='red')
axes[0].axvline(optimal_threshold, color='green', linestyle='--', linewidth=2, 
                label=f'Threshold Ótimo: {optimal_threshold:.3f}')
axes[0].axvline(0.5, color='blue', linestyle='--', linewidth=2, alpha=0.5, 
                label='Threshold Padrão: 0.5')
axes[0].set_xlabel('Threshold', fontsize=12)
axes[0].set_ylabel('Custo Total ($)', fontsize=12)
axes[0].set_title('Custo Total vs Threshold', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Plot 2: Precision-Recall vs Threshold
axes[1].plot(thresholds, precisions, linewidth=2, label='Precision', color='blue')
axes[1].plot(thresholds, recalls, linewidth=2, label='Recall', color='orange')
axes[1].axvline(optimal_threshold, color='green', linestyle='--', linewidth=2, 
                label=f'Threshold Ótimo: {optimal_threshold:.3f}')
axes[1].set_xlabel('Threshold', fontsize=12)
axes[1].set_ylabel('Score', fontsize=12)
axes[1].set_title('Precision & Recall vs Threshold', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(get_figure_path('threshold_optimization.png'))
plt.show()

print(f"\n📊 Gráfico salvo em: {get_figure_path('threshold_optimization.png')}")

## 9. Avaliação Final no OOT

Avaliamos o modelo no conjunto OOT usando o **threshold otimizado**.

In [ ]:
print("="*80)
print("AVALIAÇÃO FINAL NO OOT")
print("="*80)

# Predições com threshold otimizado
y_oot_pred_optimal = (y_oot_proba >= optimal_threshold).astype(int)

# Predições com threshold padrão (para comparação)
y_oot_pred_default = (y_oot_proba >= 0.5).astype(int)

print(f"\n📊 Resultados com Threshold Otimizado ({optimal_threshold:.3f}):")
print("\n" + classification_report(y_oot, y_oot_pred_optimal, target_names=['Normal', 'Lavagem']))

print(f"\n📊 Resultados com Threshold Padrão (0.5):")
print("\n" + classification_report(y_oot, y_oot_pred_default, target_names=['Normal', 'Lavagem']))

# Confusion Matrix
cm_optimal = confusion_matrix(y_oot, y_oot_pred_optimal)
cm_default = confusion_matrix(y_oot, y_oot_pred_default)

# Visualizar Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Threshold Otimizado
sns.heatmap(cm_optimal, annot=True, fmt='d', cmap='Blues', ax=axes[0], 
            xticklabels=['Normal', 'Lavagem'], yticklabels=['Normal', 'Lavagem'])
axes[0].set_title(f'Confusion Matrix - Threshold {optimal_threshold:.3f} (Otimizado)', 
                  fontsize=12, fontweight='bold')
axes[0].set_ylabel('Real')
axes[0].set_xlabel('Predito')

# Threshold Padrão
sns.heatmap(cm_default, annot=True, fmt='d', cmap='Oranges', ax=axes[1],
            xticklabels=['Normal', 'Lavagem'], yticklabels=['Normal', 'Lavagem'])
axes[1].set_title('Confusion Matrix - Threshold 0.5 (Padrão)', 
                  fontsize=12, fontweight='bold')
axes[1].set_ylabel('Real')
axes[1].set_xlabel('Predito')

plt.tight_layout()
plt.savefig(get_figure_path('confusion_matrices_comparison.png'))
plt.show()

print(f"\n📊 Gráfico salvo em: {get_figure_path('confusion_matrices_comparison.png')}")

## 10. Precision-Recall Curve (Métrica Adequada)

Para classes desbalanceadas, a **Precision-Recall Curve** é mais informativa que a ROC Curve.

In [ ]:
# Calcular Precision-Recall Curve
precision_curve, recall_curve, thresholds_curve = precision_recall_curve(y_oot, y_oot_proba)

# ROC Curve (para comparação)
fpr, tpr, roc_thresholds = roc_curve(y_oot, y_oot_proba)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Precision-Recall Curve
axes[0].plot(recall_curve, precision_curve, linewidth=2, color='blue')
axes[0].axhline(y=y_oot.mean(), color='red', linestyle='--', linewidth=2, 
                label=f'Baseline (No Skill): {y_oot.mean():.3f}')
axes[0].set_xlabel('Recall', fontsize=12)
axes[0].set_ylabel('Precision', fontsize=12)
axes[0].set_title(f'Precision-Recall Curve (AP: {average_precision_score(y_oot, y_oot_proba):.4f})', 
                  fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Plot 2: ROC Curve
axes[1].plot(fpr, tpr, linewidth=2, color='orange')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=2, label='Baseline (Random)')
axes[1].set_xlabel('False Positive Rate', fontsize=12)
axes[1].set_ylabel('True Positive Rate', fontsize=12)
axes[1].set_title(f'ROC Curve (AUC: {roc_auc_score(y_oot, y_oot_proba):.4f})', 
                  fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(get_figure_path('precision_recall_roc_curves.png'))
plt.show()

print(f"\n📊 Gráfico salvo em: {get_figure_path('precision_recall_roc_curves.png')}")

## 11. Salvamento do Modelo e Configurações

In [ ]:
print("="*80)
print("SALVAMENTO DO MODELO E CONFIGURAÇÕES")
print("="*80)

# Salvar modelo treinado
model_path = get_model_path(f'{best_model_name.replace(" ", "_").lower()}_final.pkl')
joblib.dump(best_model, model_path)

print(f"\n✅ Modelo salvo em: {model_path}")

# Salvar informações de treinamento
training_info = {
    'model_name': best_model_name,
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'threshold_optimal': float(optimal_threshold),
    'cost_fn': COST_FN,
    'cost_fp': COST_FP,
    'optimal_cost': float(optimal_cost),
    'cv_results': {k: {metric: float(v) for metric, v in scores.items()} 
                   for k, scores in cv_results.items()},
    'oot_metrics_optimal': {
        'precision': float(precision_score(y_oot, y_oot_pred_optimal)),
        'recall': float(recall_score(y_oot, y_oot_pred_optimal)),
        'f1': float(f1_score(y_oot, y_oot_pred_optimal)),
        'roc_auc': float(roc_auc_score(y_oot, y_oot_proba)),
        'avg_precision': float(average_precision_score(y_oot, y_oot_proba))
    },
    'oot_metrics_default': {
        'precision': float(precision_score(y_oot, y_oot_pred_default)),
        'recall': float(recall_score(y_oot, y_oot_pred_default)),
        'f1': float(f1_score(y_oot, y_oot_pred_default)),
        'roc_auc': float(roc_auc_score(y_oot, y_oot_proba)),
        'avg_precision': float(average_precision_score(y_oot, y_oot_proba))
    }
}

# Salvar JSON
info_path = get_model_path('training_info.json')
with open(info_path, 'w') as f:
    json.dump(training_info, f, indent=4)

print(f"✅ Informações de treinamento salvas em: {info_path}")

## 12. Sumário Final

### ✅ Realizações deste Notebook

1. ✅ **TimeSeriesSplit** para validação temporal (5 folds)
2. ✅ **SMOTE aplicado dentro dos folds** de CV
3. ✅ Treinamento de **5 algoritmos** (LR, RF, GB, XGBoost, LightGBM)
4. ✅ **Threshold financeiro otimizado** baseado em Matriz de Custo
5. ✅ **Precision-Recall Curve** (métrica adequada para classes desbalanceadas)
6. ✅ Avaliação no **OOT** (Out-of-Time)
7. ✅ Persistência do modelo e configurações

### 📊 Resultados Finais

**Melhor Modelo**: {best_model_name}

**Threshold Otimizado**: {optimal_threshold:.3f}

**Métricas no OOT (Threshold Otimizado)**:
- Precision: {precision_score(y_oot, y_oot_pred_optimal):.4f}
- Recall: {recall_score(y_oot, y_oot_pred_optimal):.4f}
- F1-Score: {f1_score(y_oot, y_oot_pred_optimal):.4f}
- ROC-AUC: {roc_auc_score(y_oot, y_oot_proba):.4f}
- Average Precision: {average_precision_score(y_oot, y_oot_proba):.4f}

**Economia Financeira**:
- Redução de custo vs threshold 0.5: ${default_cost - optimal_cost:,.2f}
- Percentual de redução: {(default_cost - optimal_cost)/default_cost*100:.2f}%

### 🎓 Melhores Práticas Aplicadas

✅ **Ponto 3**: Gap temporal de 7 dias entre treino e OOT (Notebook 04)  
✅ **Ponto 4**: Velocity Features com rolling windows (Notebook 05)  
✅ **Ponto 6**: Fit apenas no treino, Transform em treino e OOT (Notebook 06)  
✅ **Ponto 8**: SMOTE aplicado dentro dos folds de CV  
✅ **Ponto 9**: TimeSeriesSplit para validação temporal  
✅ **Ponto 10**: Threshold otimizado por Matriz de Custo + Precision-Recall Curve

---

**Data**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}  
**Status**: ✅ Treinamento Concluído com Sucesso